# Demo 05 : An agent loop by hand

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · the-loop

An agent is this `while` loop. Each pass sends every message so far, runs the tools the model asked for and adds the results. A reply without tool calls ends the loop. We print the input tokens of each call, which is the size of the context the model read.

In [2]:
import json
import re
import sqlite3
from pathlib import Path

from langchain.tools import tool

DB = data("game-studio/studio.db")
SECTIONS = []
for path in sorted(Path(data_folder("game-studio/docs")).glob("*.md")):
    title, *parts = path.read_text(encoding="utf-8").split("\n## ")
    SECTIONS += [f"{title.splitlines()[0]}\n## {part}" for part in parts]


def rows(sql, *args):
    with sqlite3.connect(DB) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(row) for row in conn.execute(sql, args)]


def words(text):
    return set(re.findall(r"[a-z0-9-]{3,}", text.lower()))


def best(texts, query, k):
    q = words(query)
    return sorted(texts, key=lambda t: -len(q & words(t)))[:k]

@tool
def game_info(title: str) -> str:
    """Look up a Comet Arcade game by its title or id. Returns id, title, genre, status and pitch."""
    t = title.strip()
    found = rows("SELECT game_id, title, genre, status, pitch FROM games WHERE game_id = ? OR title LIKE ?", t.upper(), f"%{t}%")
    return json.dumps(found[0]) if found else "Unknown game"

@tool
def assets_for_game(game_id: str) -> str:
    """List sprites, palettes and sounds for a game id such as CS-01, with colour and status."""
    return json.dumps(rows("SELECT asset_id, kind, name, colour, status FROM assets WHERE game_id = ?", game_id.strip().upper()))

tools = [game_info, assets_for_game]
by_name = {t.name: t for t in tools}

In [3]:
@tool
def search_design_docs(query: str) -> str:
    """Search the design docs and the studio style guide. Returns the three best matching sections."""
    return "\n\n---\n\n".join(best(SECTIONS, query, 3))

tools = [game_info, assets_for_game, search_design_docs]
by_name = {t.name: t for t in tools}
print("sections:", len(SECTIONS))

sections: 30


In [4]:
from langchain.messages import HumanMessage, SystemMessage, ToolMessage

model = llm.bind_tools(tools)
messages = [
    SystemMessage("You are the studio assistant of Comet Arcade. Use the tools. Answer in three sentences."),
    HumanMessage("Playtesters lose Comet Snake to comets they cannot see. Look up the game first, then its assets, "
                 "then the style guide section on the rule that applies. Which asset is at fault, and what should the Artist change?"),
]

calls = []
while True:
    reply = model.invoke(messages)
    messages.append(reply)
    usage = reply.usage_metadata
    calls.append((usage["input_tokens"], usage["output_tokens"]))
    names = [c["name"] for c in reply.tool_calls]
    print(f"call {len(calls)}  input {usage['input_tokens']:5}  output {usage['output_tokens']:4}  tools {names}")
    if not reply.tool_calls:
        break
    for call in reply.tool_calls:
        result = by_name[call["name"]].invoke(call["args"])
        messages.append(ToolMessage(result, tool_call_id=call["id"]))

print()
print(reply.content)

call 1  input   200  output   20  tools ['game_info']


call 2  input   283  output   22  tools ['assets_for_game']


call 3  input   585  output   32  tools ['search_design_docs']


call 4  input   832  output   93  tools []

The comet sprite (CS-SPR-03) is at fault: its blue `#3a4a7a` colour has too little contrast against the deep-space background `#0b1030`. The style guide requires game-ending hazards to reach at least 4.5:1 contrast and to use the studio danger colour. The Artist should keep the comet’s shape and change its colour to `#ff6b5e`.


## b · the-bill

Every call is billed for the whole context again. The total billed input is larger than the context of the last call.

In [5]:
print("model calls:", len(calls))
print("context of the last call:", calls[-1][0], "tokens")
print("input tokens billed in total:", sum(i for i, _ in calls))
print("output tokens in total:", sum(o for _, o in calls))
# gpt-6-luna on OpenRouter : $0.10 per million input tokens, $0.50 per million output tokens
cost = (sum(i for i, _ in calls) * 0.10 + sum(o for _, o in calls) * 0.50) / 1_000_000
print(f"cost at list price: ${cost:.6f}")

model calls: 4
context of the last call: 832 tokens
input tokens billed in total: 1900
output tokens in total: 167
cost at list price: $0.000273


## c · what-fills-the-context

We count each kind of message with the tokenizer of demo01.

In [6]:
import tiktoken

encoding = tiktoken.get_encoding("o200k_base")
size = {}
for m in messages:
    text = m.content if isinstance(m.content, str) else str(m.content)
    size[m.type] = size.get(m.type, 0) + len(encoding.encode(text + str(getattr(m, "tool_calls", ""))))
size["tool definitions"] = len(encoding.encode(str([t.args_schema.model_json_schema() for t in tools])))
for kind, tokens in sorted(size.items(), key=lambda kv: -kv[1]):
    print(f"{kind:18} {tokens:6}")

tool                  523
ai                    253
tool definitions      195
human                  48
system                 19


## Things to try

Optional : we can change the sentence or the question and run the example again.